In [0]:
dbutils.library.restartPython()

In [0]:
from pyspark.sql import functions as F

CAMINHO = "/Volumes/voebem/bronze/arquivos/vra/*.csv"
TABELA = "voebem.bronze.vra"

## Config dos .csv:
- sep: ";"
- header: True
- skipRows: 1 (Pular linha "Atualizado em: ...)
- inferSchema *desligado*

In [0]:
bruto = (
    spark.read.format("csv")
    .option("sep", ";")
    .option("header", True)
    .option("skipRows", 1)
    .option("quote", "")
    .option("escape", "")
    .option("encoding", "UTF-8")
    .option("mode", "PERMISSIVE")
    .load(CAMINHO)
)


## Normalizar nomes
Não podemos ter **espaços** nos nomes das colunas (ex. "ICAO Empresa Aérea").
- Transformar espaços em "_"
- Usar lowercase nos nomes

In [0]:
normalize_table = str.maketrans({"ã": "a", "á": "a", "à": "a", "â": "a", "é": "e", "ê": "e", "í": "i", "ì": "i", "î": "i", "ó": "o", "ò": "o", "ô": "o", "õ": "o", "ú": "u", "ù": "u", "û": "u", "ç": "c", " ": "_", "(": "", ")": "", '"': ""})

for cols in bruto.columns:
    novo = cols.translate(normalize_table).lower()
    print(f"\n{cols!r} -> {novo!r}")
    bruto = bruto.withColumnRenamed(cols, novo)

In [0]:
# Adicionar metadados
bronze = bruto.withColumn(
    "_arquivo_origem", F.col("_metadata.file_name")
).withColumn(
    "_ingerido_em", F.current_timestamp()
)

In [0]:
bronze.write.format("delta").mode("overwrite").options(overwriteSchema="true").saveAsTable(TABELA)

print(f"{TABELA}: {spark.table(TABELA).count()} linhas")

In [0]:
spark.sql(f"""
          COMMENT ON TABLE {TABELA} IS 
          'Bronze - VRA (Voo Regular Ativo) da ANAC, 12 meses (2025-08 a 2026-07).
          Dado bruto: todas as colunas são strings, nenhuma linha descartada.
          Carga full refresh idempotente a partir de /Volumes/voebem/bronze/arquivos/vra/'
          """)

In [0]:
display(
    spark.sql(f"""
              SELECT _arquivo_origem, COUNT(*) AS linhas, MAX(_ingerido_em) AS ingerido_em
              FROM {TABELA}
              GROUP BY _arquivo_origem
              ORDER BY _arquivo_origem
              """)
)